In [1]:
!pip install pyspark

In [2]:
with open("/content/sensor_logs.csv", "w") as f:
    f.write("""device_id,timestamp,energy_kwh
1,2026-05-13 08:00:00,2.5
1,2026-05-13 19:00:00,3.5
2,2026-05-13 09:00:00,5.2
2,2026-05-13 20:00:00,4.8
3,2026-05-13 10:00:00,1.1
3,2026-05-13 21:00:00,2.0
""")

print("sensor_logs.csv created!")

sensor_logs.csv created!


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, hour, sum, when

# Create Spark session
spark = SparkSession.builder.appName("Smart Home Energy Analysis").getOrCreate()

# Load CSV
df = spark.read.csv("/content/sensor_logs.csv", header=True, inferSchema=True)

print("Original Data:")
df.show()

# Extract hour
df = df.withColumn("hour", hour(col("timestamp")))

# Peak vs Off-peak
df = df.withColumn(
    "usage_type",
    when((col("hour") >= 18) & (col("hour") <= 22), "Peak")
    .otherwise("Off-Peak")
)

# Aggregate by device and usage type
usage_summary = df.groupBy("device_id", "usage_type") \
    .agg(sum("energy_kwh").alias("total_energy"))

print("Peak vs Off-Peak Usage:")
usage_summary.show()

# Top energy-consuming devices
top_devices = df.groupBy("device_id") \
    .agg(sum("energy_kwh").alias("total_energy")) \
    .orderBy(col("total_energy").desc())

print("Top Energy-Consuming Devices:")
top_devices.show()

spark.stop()

Original Data:
+---------+-------------------+----------+
|device_id|          timestamp|energy_kwh|
+---------+-------------------+----------+
|        1|2026-05-13 08:00:00|       2.5|
|        1|2026-05-13 19:00:00|       3.5|
|        2|2026-05-13 09:00:00|       5.2|
|        2|2026-05-13 20:00:00|       4.8|
|        3|2026-05-13 10:00:00|       1.1|
|        3|2026-05-13 21:00:00|       2.0|
+---------+-------------------+----------+

Peak vs Off-Peak Usage:
+---------+----------+------------+
|device_id|usage_type|total_energy|
+---------+----------+------------+
|        2|  Off-Peak|         5.2|
|        1|      Peak|         3.5|
|        2|      Peak|         4.8|
|        3|  Off-Peak|         1.1|
|        1|  Off-Peak|         2.5|
|        3|      Peak|         2.0|
+---------+----------+------------+

Top Energy-Consuming Devices:
+---------+------------+
|device_id|total_energy|
+---------+------------+
|        2|        10.0|
|        1|         6.0|
|        3|   